# Transfer learning with MobileNetV2

**Learning objective:** Use an ImageNet-pretrained TensorFlow backbone as a frozen feature extractor and inspect the resulting representation.

This notebook is part of the TensorFlow/Keras learning track. It is designed to be read top-to-bottom: intuition → shapes → mathematics → TensorFlow implementation → observed result → interpretation.

> GitHub renders the committed executed output as a static learning artifact. Clone the repository and rerun it in Jupyter/VS Code for live experimentation.


In [1]:
import os, warnings, random
from pathlib import Path
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow:", tf.__version__)
print("Keras:", tf.keras.__version__ if hasattr(tf.keras, "__version__") else "bundled with TensorFlow")
print("Execution device(s):", [d.device_type for d in tf.config.list_logical_devices()])


2026-09-21 07:16:16.904652: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1789974976.921144    4195 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1789974976.925549    4195 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


TensorFlow: 2.18.1
Keras: 3.15.1
Execution device(s): ['CPU']


2026-09-21 07:16:18.707425: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Transfer learning reuses representations learned from a large source task. We freeze a pretrained MobileNetV2 backbone and transform a Fashion-MNIST image into a 1,280-dimensional feature vector.


In [2]:
base=tf.keras.applications.MobileNetV2(weights="imagenet",include_top=False,input_shape=(96,96,3),pooling="avg")
base.trainable=False
(x,y),_=tf.keras.datasets.fashion_mnist.load_data(); img=tf.cast(x[0][...,None],tf.float32)
img=tf.image.resize(img,(96,96)); img=tf.repeat(img,3,axis=-1); batch=tf.keras.applications.mobilenet_v2.preprocess_input(img[None,...])
feat=base(batch,training=False)
print("backbone trainable:",base.trainable,"parameters:",base.count_params()); print("feature vector:",feat.shape,"L2 norm:",round(float(tf.norm(feat)),3))


      0/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0s/step

5513216/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


backbone trainable: False parameters: 2257984
feature vector: (1, 1280) L2 norm: 39.695


In [3]:
head=tf.keras.Sequential([tf.keras.layers.Input((1280,)),tf.keras.layers.Dense(10,activation="softmax")])
print("small task-specific head parameters:",head.count_params())


small task-specific head parameters: 12810


Typical workflow: train the new head with the backbone frozen, then optionally unfreeze a small upper portion with a much lower learning rate. Validation data decides whether fine-tuning helps.
